# Task 8: Integrating Streamlit Frontend with Flask REST API
### Real-Time Deep Learning Clinical Risk Prediction Platform
**Course / Platform**: L&T Edutech Deep Learning Deployment

---

## 1. Objective & Scope
The primary goal of Task 8 is to establish robust, real-time client-server communication between:
1. **Flask REST API (Backend)**: Serves a pre-trained 3-layer PyTorch Artificial Neural Network (`DeepHealthRiskNet`).
2. **Streamlit Multi-Page UI (Frontend)**: Ingests patient parameters, constructs structured HTTP request payloads, communicates over REST API, and dynamically renders predictions and analytics.

### Key Approach Steps:
- **Connect**: Establish HTTP communication via Python `requests` library.
- **Transfer**: Send patient feature vectors via HTTP `POST /predict` endpoints.
- **Predict**: Execute PyTorch forward pass in backend Flask service.
- **Display**: Render confidence gauge, risk categories, probability distributions, and latency metrics in Streamlit.
- **Validate**: Execute 11-step automated integration test suite.

## 2. Environment Setup & Component Verification

In [1]:
import os
import sys
import json
import time
import requests
import torch
import numpy as np
import pandas as pd

# Verify PyTorch engine and dataset availability
from model_loader import get_model_engine
from utils import ModelConnector, load_dataset, FEATURE_NAMES

engine = get_model_engine()
df = load_dataset()
print(f"PyTorch Model Engine Loaded: {engine.config['model_name']} on device {engine.device}")
print(f"Loaded Dataset: {len(df)} patient records with {len(FEATURE_NAMES)} features.")

## 3. Flask REST API Backend Architecture

In [2]:
# Inspect Flask API routes
from flask_api import app

print("Flask API Endpoints:")
for rule in app.url_map.iter_rules():
    print(f"- {rule.methods} {rule.rule}")

## 4. Single Patient Real-Time HTTP API Inference Test

In [3]:
# Execute test request via Flask Test Client
with app.test_client() as client:
    payload = {
        "features": [45.0, 170.0, 75.0, 25.95, 6500.0, 2400.0, 6.8, 78.0, 128.0, 82.0, 2.5, 4.0, 0.0, 0.0]
    }
    response = client.post("/predict", json=payload)
    print("HTTP Status Code:", response.status_code)
    print("JSON Response:")
    print(json.dumps(response.get_json(), indent=2))

## 5. Cohort CSV Batch API Processing Test

In [4]:
# Ingest sample cohort and run batch inference
sample_cohort_path = "sample_cohort_data.csv"
df_cohort = pd.read_csv(sample_cohort_path)
print(f"Sample Cohort Records: {len(df_cohort)}")

# Connector test
connector = ModelConnector()
results = engine.predict([df.iloc[i][["Age", "Height_cm", "Weight_kg", "BMI", "Daily_Steps", "Calories_Intake", "Hours_of_Sleep", "Heart_Rate", "Systolic_BP", "Diastolic_BP", "Exercise_Hours_per_Week", "Alcohol_Consumption_per_Week", "Smoker", "Diabetic"]].map(lambda v: 1.0 if str(v).lower() in ["yes", "true"] else (0.0 if str(v).lower() in ["no", "false"] else float(v))).tolist() for i in range(min(5, len(df)))])

print(f"Batch Predictions Count: {len(results)}")
for r in results[:3]:
    print(f"Sample #{r['sample_index']}: {r['predicted_label']} ({r['confidence_score']*100:.1f}%) probabilities: {r['class_probabilities']}")

## 6. Automated Integration Test Suite Execution

In [5]:
import unittest
from test_integration import Task8IntegrationTestSuite

suite = unittest.TestLoader().loadTestsFromTestCase(Task8IntegrationTestSuite)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 7. Summary & Final Deliverables
- **Integrated Application**: Streamlit Multi-Page UI (`app.py`, `views/`) + Flask REST API (`flask_api.py`).
- **API Capabilities**: Real-time single & batch neural network forward passes, HTTP error handlers, health probes.
- **Verification**: 100% test pass rate across 11 integration test cases.
- **Documentation & PDF**: Complete PDF report generated with source code, test benchmarks, and architectural specs.